In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [53]:
df = pd.read_csv('../data/raw/Works Completed.csv')

In [54]:
df.head()

,Sr. No.,Work Category,Work,State,IDA,Work Description,Hon'ble Members of Parliament,Constituency,Image,Completion Date,Amount Disbursed ( ₹ )
0,1,Normal/Others,WS/MP418/2024-2025/133409-Construction of road...,Bihar,ARARIA(DISTRICT PLANNING OFFICER ARARIA_IDA),PCC Road from Permeshwar Bhagat house to Ramde...,Pradeep Kumar Singh,ARARIA,NaN,05-Sep-2024,448127
1,2,Normal/Others,WS/MP18152/2024-2025/133691-Construction of ro...,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),NaN,07-Apr-2025,300000
2,3,Normal/Others,WS/MP345/2024-2025/134140-Construction of road...,Kerala,KOLLAM(DISTRICT COLLECTOR KOLLAM_IDA),Concreting of road from Janathavayanasala - P...,Shri NK Premachandran,KOLLAM,NaN,12-Aug-2024,293492
3,4,Normal/Others,WS/MP18152/2024-2025/133686-Construction of ro...,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),NaN,07-Apr-2025,300000
4,5,Normal/Others,WS/MP18152/2024-2025/133690-Construction of ro...,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),NaN,07-Apr-2025,300000


In [55]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16001 entries, 0 to 16000
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   Sr. No.                        16001 non-null  str  
 1   Work Category                  16001 non-null  str  
 2   Work                           16001 non-null  str  
 3   State                          16001 non-null  str  
 4   IDA                            16001 non-null  str  
 5   Work Description               15946 non-null  str  
 6   Hon'ble Members of Parliament  16001 non-null  str  
 7   Constituency                   16001 non-null  str  
 8   Image                          10885 non-null  str  
 9   Completion Date                16001 non-null  str  
 10  Amount Disbursed ( ₹ )         15994 non-null  str  
dtypes: str(11)
memory usage: 1.3 MB


In [56]:
df.isnull().sum()

Sr. No.                             0
Work Category                       0
Work                                0
State                               0
IDA                                 0
Work Description                   55
Hon'ble Members of Parliament       0
Constituency                        0
Image                            5116
Completion Date                     0
Amount Disbursed ( ₹ )              7
dtype: int64

In [57]:
df.duplicated().sum()

0

In [58]:
df.columns = (
    pd.Index(df.columns)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

In [59]:
df.columns

Index(['sr_no', 'work_category', 'work', 'state', 'ida', 'work_description',
       'hon_ble_members_of_parliament', 'constituency', 'image',
       'completion_date', 'amount_disbursed'],
      dtype='str')

In [60]:
for col in df.columns:
    if col == "work":
        continue

    df[col] = (
        df[col]
        .astype("string")
        .str.replace("\u00a0", " ", regex=False)
        .str.replace(r"[\t\r\n]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


In [61]:
grand_total_mask = pd.Series(False, index=df.index)

for col in df.columns:
    if col == "work":
        continue

    grand_total_mask |= df[col].str.fullmatch(
        r"grand\s+total",
        case=False,
        na=False
    )

print("Grand Total rows found:", int(grand_total_mask.sum()))

df = df.loc[~grand_total_mask].copy()

Grand Total rows found: 1


In [62]:
df = df.replace(r"^\s*$", pd.NA, regex=True)

print("Rows remaining:", len(df))

Rows remaining: 16000


In [63]:
df["work"] = df["work"].str.replace(
    r"\s*/\s*",
    "/",
    regex=True
)

In [64]:
df["work_id"] = df["work"].str.extract(
    r"(?i)(WS/MP\d+/\d{4}-\d{4}/\d+)",
    expand=False
)
df["work_id"] = df["work_id"].str.upper()

In [65]:
df["work_detail_from_reference"] = df["work"].str.extract(
    r"(?i)^WS/MP\d+/\d{4}-\d{4}/\d+-(.*)$",
    expand=False
).str.strip()

In [66]:
df.head()

,sr_no,work_category,work,state,ida,work_description,hon_ble_members_of_parliament,constituency,image,completion_date,amount_disbursed,work_id,work_detail_from_reference
0,1,Normal/Others,WS/MP418/2024-2025/133409-Construction of road...,Bihar,ARARIA(DISTRICT PLANNING OFFICER ARARIA_IDA),PCC Road from Permeshwar Bhagat house to Ramde...,Pradeep Kumar Singh,ARARIA,<NA>,05-Sep-2024,448127,WS/MP418/2024-2025/133409,"Construction of roads, link roads, pathways or..."
1,2,Normal/Others,WS/MP18152/2024-2025/133691-Construction of ro...,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),<NA>,07-Apr-2025,300000,WS/MP18152/2024-2025/133691,Construction of rooms and halls in school and ...
2,3,Normal/Others,WS/MP345/2024-2025/134140-Construction of road...,Kerala,KOLLAM(DISTRICT COLLECTOR KOLLAM_IDA),Concreting of road from Janathavayanasala - Pa...,Shri NK Premachandran,KOLLAM,<NA>,12-Aug-2024,293492,WS/MP345/2024-2025/134140,"Construction of roads, link roads, pathways or..."
3,4,Normal/Others,WS/MP18152/2024-2025/133686-Construction of ro...,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),<NA>,07-Apr-2025,300000,WS/MP18152/2024-2025/133686,Construction of rooms and halls in school and ...
4,5,Normal/Others,WS/MP18152/2024-2025/133690-Construction of ro...,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),<NA>,07-Apr-2025,300000,WS/MP18152/2024-2025/133690,Construction of rooms and halls in school and ...


In [67]:
amount_text = (
    df["amount_disbursed"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.replace("₹", "", regex=False)
    .str.strip()
)


In [68]:
missing_amount = amount_text.str.upper().isin(
    ["", "NA", "N/A", "NULL", "-"]
)
amount_text = amount_text.mask(missing_amount, pd.NA)

In [69]:
df["amount_disbursed"] = pd.to_numeric(
    amount_text,
    errors="coerce"
)

In [70]:
def parse_date_column(series):
    cleaned = series.astype("string").str.strip()

    missing = cleaned.str.upper().isin(
        ["", "NA", "N/A", "NULL", "-"]
    )
    cleaned = cleaned.mask(missing, pd.NA)

    return pd.to_datetime(
        cleaned,
        format="%d-%b-%Y",
        errors="coerce"
    )


In [71]:
raw_dates = df['completion_date'].astype("string").str.strip()
missing = raw_dates.str.upper().isin(
    ["", "NA", "N/A", "NULL", "-"]
)
parsed_dates = parse_date_column(raw_dates)

df['completion_date'] = parsed_dates

In [72]:
df.drop(columns=["work"], inplace=True)

In [73]:
df["image"] = (
    df["image"]
    .astype("string")
    .str.strip()
)

df["image_marker_present"] = (
    df["image"]
    .fillna("")
    .str.casefold()
    .eq("images")
    .fillna(False)
    .astype("int8")
)


In [74]:
df.head()

,sr_no,work_category,state,ida,work_description,hon_ble_members_of_parliament,constituency,image,completion_date,amount_disbursed,work_id,work_detail_from_reference,image_marker_present
0,1,Normal/Others,Bihar,ARARIA(DISTRICT PLANNING OFFICER ARARIA_IDA),PCC Road from Permeshwar Bhagat house to Ramde...,Pradeep Kumar Singh,ARARIA,<NA>,2024-09-05,448127.0,WS/MP418/2024-2025/133409,"Construction of roads, link roads, pathways or...",0
1,2,Normal/Others,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),<NA>,2025-04-07,300000.0,WS/MP18152/2024-2025/133691,Construction of rooms and halls in school and ...,0
2,3,Normal/Others,Kerala,KOLLAM(DISTRICT COLLECTOR KOLLAM_IDA),Concreting of road from Janathavayanasala - Pa...,Shri NK Premachandran,KOLLAM,<NA>,2024-08-12,293492.0,WS/MP345/2024-2025/134140,"Construction of roads, link roads, pathways or...",0
3,4,Normal/Others,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),<NA>,2025-04-07,300000.0,WS/MP18152/2024-2025/133686,Construction of rooms and halls in school and ...,0
4,5,Normal/Others,Punjab,FARIDKOT(DEPUTY COMMISSIONER FARIDKOT_IDA),Construction of MID DAY Meal Shed in Govt Prim...,SARABJEET SINGH KHALSA,FARIDKOT(SC),<NA>,2025-04-07,300000.0,WS/MP18152/2024-2025/133690,Construction of rooms and halls in school and ...,0


In [78]:
df.to_csv('../data/processed/works_completed_cleaned.csv', index=False)